# 03 – Cells and ligand–receptor links for the XCL1 run (CD8αα(I)/NK → DC1/aDC1)

**Question.** Which cells and which L–R links go into the CausalCCC run?

Same rules as thymus case 1 (notebook 04):
- merged groups: senders = CD8αα(I) + NK (`cd8aa_nk`), receivers = DC1 + aDC1 (`dc1`), population as a variable;
- at most **500 cells per population** (random, fixed seed), so that no population dominates its group;
- L–R links: the claimed pairs (XCL1 → XCR1, XCL2 → XCR1) are always included; the others are ranked by the
  **S5 mean** (notebook 02), up to 25 S5 pairs;
- clean-up while filling the list: a pair is skipped if all its gene links are already in; pairs whose "receptor" is
  IDE (an enzyme) or whose "ligand" is LCK (an intracellular kinase, not a signalling ligand) are excluded;
- metadata variables: population, donor, chemistry, **stage** (all ages are pooled here) and **sequencing depth**
  (`n_counts`, total raw counts per cell, computed by the authors). Depth was added after a first run without it, as
  in Minh's runs A and B (`notebooks/thymus_early/`): housekeeping genes whose counts follow the depth of the cell
  (GAPDH, ACTB, SERF2, PPIA, EEF1A1, TPT1, PTMA) were the hubs of the sender network.

This notebook only prepares the inputs; the CausalCCC steps run in R (notebook 04). The counts are exported in Matrix
Market format.

In [1]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import anndata as ad
import scipy.io

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
RES = ROOT / "results/thymus_case2_xcl1"
OUT = RES / "03_run_input"
OUT.mkdir(parents=True, exist_ok=True)

CAP = 500
SEED = 0
N_LINKS = 25                      # L-R pairs (S5 rows); CausalCCC networks usually have about 20-30 links
METADATA = ["population", "donor", "method", "stage", "n_counts"]
# Receiver-side genes of interest added in notebook 04: DC1 activation and migration (notebook 01, section 1)
ACTIVATION_GENES = ["CCR7", "LAMP3", "CD40", "CD80", "CD86", "IL12B"]
MAX_PIVOTS = 30                   # causalCCC.MIselection(): at most 30 genes of interest per group

## 1. Cap the cells at 500 per population

In [2]:
cells = pd.read_csv(RES / "01_cell_populations/selected_cells.csv", index_col=0)
capped = pd.concat([d.sample(n=min(len(d), CAP), random_state=SEED) for _, d in cells.groupby("population")])

summary = pd.DataFrame({"before": cells.population.value_counts(), "after": capped.population.value_counts()})
summary["group"] = cells.groupby("population").group.first()
summary.sort_values(["group", "after"], ascending=[True, False])

,before,after,group
population,,,
CD8αα(I),3899,500,cd8aa_nk
NK,1615,500,cd8aa_nk
DC1,1383,500,dc1
aDC1,135,135,dc1


In [3]:
print(capped.group.value_counts().to_string())
# Stage and donor mix per group, after the cap
pd.concat({"stage": pd.crosstab([capped.group, capped.population], capped.stage)[["early", "middle", "late"]],
           "chemistry": pd.crosstab([capped.group, capped.population], capped.method)}, axis=1)

group
cd8aa_nk    1000
dc1          635


stage             chemistry     
                    early middle late      3GEX 5GEX
group    population                                 
cd8aa_nk CD8αα(I)      45    226  229       176  324
         NK           310    117   73       322  178
dc1      DC1          136    298   66       298  202
         aDC1          20     97   18        40   95

**Reading.** The senders are 1,000 cells (CD8αα(I) and NK, 500 each) and the receivers 635 (DC1 500, aDC1 135, so
aDC1 is about a fifth of the group). The stage link from notebook 01 survives the cap: NK cells are mostly early
(310 of 500), CD8αα(I) cells mostly middle or late (455 of 500).

## 2. Choose the L–R links

CausalCCC links **genes**: a CellPhoneDB complex is replaced by its subunit genes, and the ligand is linked to each.

In [4]:
DB = THY / "cellphonedb"
gene = pd.read_csv(DB / "gene_input.csv").drop_duplicates("uniprot").set_index("uniprot")
gene["gene_name"] = gene.gene_name.replace({"NECTIN1": "PVRL1", "NECTIN2": "PVRL2", "NECTIN3": "PVRL3"})
cplx = pd.read_csv(DB / "complex_input.csv").set_index("complex_name")

def genes_of(name):
    if name in cplx.index:
        units = cplx.loc[name, ["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4"]].dropna()
        return [gene.gene_name[u] for u in units]
    return [name]                                    # already a gene symbol (notebook 02)

cand = pd.read_csv(RES / "02_ligand_receptor_candidates/candidates_cd8aa_nk_to_dc1.csv")
CLAIMED = ["XCL1_XCR1", "XCL2_XCR1"]
EXCLUDED_RECEPTORS = {"IDE": "receptor is an enzyme (IDE)"}
EXCLUDED_LIGANDS = {"LCK": "ligand is an intracellular kinase (LCK)"}
assert set(CLAIMED) <= set(cand.pair)

cand["claimed"] = cand.pair.isin(CLAIMED)
ranked = cand.sort_values(["claimed", "s5_max_mean"], ascending=[False, False]).reset_index(drop=True)

chosen, skipped, gene_links = [], [], set()
for _, row in ranked.iterrows():
    if len(chosen) == N_LINKS:
        break
    lig, rec = genes_of(row.ligand), genes_of(row.receptor)
    new_links = {(l, r) for l in lig for r in rec}
    if set(rec) & set(EXCLUDED_RECEPTORS):
        skipped.append((row.pair, EXCLUDED_RECEPTORS[(set(rec) & set(EXCLUDED_RECEPTORS)).pop()]))
    elif set(lig) & set(EXCLUDED_LIGANDS):
        skipped.append((row.pair, EXCLUDED_LIGANDS[(set(lig) & set(EXCLUDED_LIGANDS)).pop()]))
    elif new_links <= gene_links:
        skipped.append((row.pair, "all gene links already included"))
    else:
        chosen.append(row.pair)
        gene_links |= new_links

links = ranked[ranked.pair.isin(chosen)].copy()
cols = ["pair", "ligand", "receptor", "orientation_rule", "claimed", "s5_max_mean", "s5_best_pair",
        "max_ligand_frac", "ligand_pops", "max_receptor_frac", "receptor_pops"]
print("skipped while filling the list:", skipped)
links[cols].round(2)

skipped while filling the list: [('CCL3_IDE', 'receptor is an enzyme (IDE)'), ('CD8 receptor_LCK', 'ligand is an intracellular kinase (LCK)')]


,pair,ligand,receptor,orientation_rule,claimed,s5_max_mean,s5_best_pair,max_ligand_frac,ligand_pops,max_receptor_frac,receptor_pops
0,XCL1_XCR1,XCL1,XCR1,receptor flag,True,1.12,CD8αα(I) -> DC1,0.67,"CD8αα(I), NK",0.56,DC1
1,XCL2_XCR1,XCL2,XCR1,receptor flag,True,0.91,NK -> DC1,0.84,NK,0.56,DC1
2,CD74_MIF,MIF,CD74,receptor flag,False,3.86,CD8αα(I) -> DC1,0.72,"CD8αα(I), NK",1.00,"DC1, aDC1"
3,APP_CD74,APP,CD74,receptor flag,False,2.90,CD8αα(I) -> DC1,0.22,"CD8αα(I), NK",1.00,"DC1, aDC1"
4,MIF_TNFRSF14,MIF,TNFRSF14,receptor flag,False,1.42,CD8αα(I) -> aDC1,0.72,"CD8αα(I), NK",0.59,"DC1, aDC1"
5,LGALS9_CD44,LGALS9,CD44,receptor flag,False,1.11,NK -> aDC1,0.12,CD8αα(I),0.80,"DC1, aDC1"
6,CLEC2B_KLRF1,KLRF1,CLEC2B,ambiguous,False,0.86,NK -> DC1,0.46,NK,0.26,DC1
7,CD28_CD86,CD28,CD86,receptor flag,False,0.84,CD8αα(I) -> aDC1,0.54,CD8αα(I),0.63,"DC1, aDC1"
8,CD40LG_CD40,CD40LG,CD40,receptor flag,False,0.81,CD8αα(I) -> aDC1,0.11,CD8αα(I),0.90,"DC1, aDC1"
10,IFNG_Type II IFNR,IFNG,Type II IFNR,receptor flag,False,0.78,NK -> aDC1,0.43,NK,0.50,"DC1, aDC1"


In [5]:
# Where the cut falls: the next candidates that were not included
last = ranked.index[ranked.pair == chosen[-1]][0]
ranked.iloc[last + 1:last + 7][["pair", "orientation_rule", "s5_max_mean", "s5_best_pair"]].round(2)

,pair,orientation_rule,s5_max_mean,s5_best_pair
27,PDCD1LG2_PDCD1,ambiguous,0.43,CD8αα(I) -> aDC1
28,FLT3LG_FLT3,receptor flag,0.42,NK -> aDC1
29,aLb2 complex_F11R,receptor flag,0.42,NK -> aDC1
30,TNF_TNFRSF1B,receptor flag,0.41,NK -> aDC1
31,BTLA_TNFRSF14,ambiguous,0.40,NK -> DC1
32,ALCAM_CD6,ambiguous,0.39,CD8αα(I) -> aDC1


In [6]:
edges = pd.DataFrame([dict(ligands=l, receptors=r, pair=row.pair)
                      for _, row in links.iterrows()
                      for l in genes_of(row.ligand) for r in genes_of(row.receptor)]).drop_duplicates(["ligands", "receptors"])
ligand_genes, receptor_genes = sorted(edges.ligands.unique()), sorted(edges.receptors.unique())
receiver_pivots = sorted(set(receptor_genes) | set(ACTIVATION_GENES))
print(len(links), "L-R pairs ->", len(edges), "gene-gene links")
print("ligand genes:  ", len(ligand_genes), ligand_genes)
print("receptor genes:", len(receptor_genes), receptor_genes)
print("receiver pivots (receptors + activation genes):", len(receiver_pivots), "| limit", MAX_PIVOTS)
# The metadata are not pivots in notebook 04 (metadata_list = NULL), so only genes count towards the limit
assert len(ligand_genes) <= MAX_PIVOTS and len(receiver_pivots) <= MAX_PIVOTS

25 L-R pairs -> 28 gene-gene links
ligand genes:   24 ['APP', 'CD200', 'CD28', 'CD38', 'CD40LG', 'CD47', 'CRTAM', 'ICAM3', 'IFNG', 'ITGA4', 'ITGAL', 'ITGB1', 'ITGB2', 'KLRF1', 'LGALS9', 'LTB', 'MIF', 'PDCD1', 'SELL', 'SPN', 'TIGIT', 'TNFSF14', 'XCL1', 'XCL2']
receptor genes: 22 ['CADM1', 'CD200R1', 'CD274', 'CD40', 'CD44', 'CD74', 'CD80', 'CD86', 'CLEC2B', 'ICAM1', 'IFNGR1', 'IFNGR2', 'ITGAL', 'ITGB2', 'LTBR', 'PECAM1', 'PLAUR', 'PVRL2', 'SELPLG', 'SIRPA', 'TNFRSF14', 'XCR1']
receiver pivots (receptors + activation genes): 25 | limit 30


In [7]:
# Genes that are both a ligand and a receptor in this set (CausalCCC suffixes them _senders / _receivers)
sorted(set(ligand_genes) & set(receiver_pivots))

['ITGAL', 'ITGB2']

**Reading.**
- 25 S5 pairs = the 2 claimed XCL pairs + 23 by S5 mean, giving 28 gene–gene links (complexes such as the IFN-γ
  receptor, LFA-1 = `aLb2 complex` and VLA-4 = `a4b1 complex` are split into their subunits).
- Skipped while filling: `CCL3_IDE` (IDE is an enzyme) and `CD8 receptor_LCK` (LCK is an intracellular kinase).
- Genes of interest: 24 ligand genes; 22 receptor genes, plus CCR7, LAMP3 and IL12B (CD40, CD80 and CD86 are already
  receptors) = 25 receiver pivots, within the limit of 30.
- ITGAL and ITGB2 (LFA-1) are on both sides: as a ligand complex on CD8αα(I)/NK cells binding ICAM1 on DCs, and as a
  receptor on DCs binding ICAM3. CausalCCC gives them `_senders` / `_receivers` suffixes.
- The cut falls just before PDCD1LG2–PDCD1, FLT3LG → FLT3 (a DC growth factor) and TNF → TNFRSF1B (S5 mean 0.41–0.43).

## 3. Export for R (notebook 04)

- `counts.mtx.gz`: raw counts, genes × cells (all 33,694 genes, because the gene selection ranks every gene);
- `genes.csv`, `cells.csv`: row and column names; `cells.csv` also holds the group and the metadata;
- `interact_edges.tsv`: the gene–gene L–R links (`ligands`, `receptors`);
- `selected_links.csv`: the 25 S5 pairs with their ranking information.

Duplicated gene symbols get a `-1` suffix (`var_names_make_unique`); none of them is a ligand, receptor or
activation gene.

In [8]:
raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")
rows = np.sort(raw.obs_names.get_indexer(capped.index))
assert (rows >= 0).all()
sub = raw[rows].to_memory()
sub.var_names_make_unique()
assert set(ligand_genes + receiver_pivots) <= set(sub.var_names)

with gzip.open(OUT / "counts.mtx.gz", "wb") as f:
    scipy.io.mmwrite(f, sub.X.T.tocoo().astype(np.int32), field="integer")
pd.Series(sub.var_names, name="gene").to_csv(OUT / "genes.csv", index=False)
capped.loc[sub.obs_names, ["group"] + METADATA + ["Age"]].rename_axis("cell").to_csv(OUT / "cells.csv")
edges[["ligands", "receptors"]].to_csv(OUT / "interact_edges.tsv", sep="\t", index=False)
links[cols].to_csv(OUT / "selected_links.csv", index=False)
print(sub.shape, "->", sorted(p.name for p in OUT.iterdir()))

/tmp/ipykernel_417/2870553642.py:1: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


/tmp/ipykernel_417/2870553642.py:4: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  sub = raw[rows].to_memory()


(1635, 33694) -> ['cells.csv', 'counts.mtx.gz', 'genes.csv', 'interact_edges.tsv', 'selected_links.csv']


## Summary

| | |
|---|---|
| Senders (`cd8aa_nk`) | 1,000 cells: CD8αα(I) 500 · NK 500 |
| Receivers (`dc1`) | 635 cells: DC1 500 · aDC1 135 |
| L–R pairs | 25 S5 pairs = XCL1 → XCR1, XCL2 → XCR1 + 23 by S5 mean → 28 gene–gene links |
| Skipped while filling | `CCL3_IDE` (enzyme), `CD8 receptor_LCK` (intracellular kinase) |
| Genes of interest | 24 ligands; 22 receptors + 3 activation genes (CCR7, LAMP3, IL12B) = 25 |
| Metadata variables | population, donor, chemistry, stage, sequencing depth (`n_counts`) |
| Exported to | `results/thymus_case2_xcl1/03_run_input/` |

**Points to keep in mind when reading the network:**
- XCL1 in the sender group comes mostly from NK cells (notebook 01), and NK ≈ early stage: an XCL1–population link
  will be hard to tell apart from an XCL1–stage link.
- XCR1 (DC1) and CCR7 (aDC1) mark two states of the receiver group; the claim predicts that they are connected through
  `population`, not directly.
- Ranking by S5 mean again favours ubiquitous genes (MIF, APP → CD74), as in thymus case 1.